In [20]:
from pathlib import Path
import numpy as np
import json

# Détecter la racine du projet COFI
candidats = [Path.cwd(), *Path.cwd().parents]

RACINE_PROJET = next(
    (
        dossier for dossier in candidats
        if (dossier / "data").is_dir()
        and (dossier / "notebooks").is_dir()
    ),
    None
)

if RACINE_PROJET is None:
    raise FileNotFoundError("Racine du projet COFI introuvable.")

DOSSIER_PROCESSED = RACINE_PROJET / "data" / "processed"

# Charger les paramètres
with open(
    DOSSIER_PROCESSED / "parametres_preparation.json",
    "r",
    encoding="utf-8"
) as fichier:
    parametres = json.load(fichier)

# Charger le vocabulaire
with open(
    DOSSIER_PROCESSED / "vocabulaire.json",
    "r",
    encoding="utf-8"
) as fichier:
    token_vers_id = json.load(fichier)

# Paramètres du projet
VOCAB_SIZE = len(token_vers_id)

MAX_LEN_ENCODER = parametres["longueur_max_encodeur"]
MAX_LEN_TARGET = parametres["longueur_max_cible_complete"]
MAX_LEN_DECODER = parametres["longueur_decodeur"]

PAD_ID = 0
UNK_ID = 1
SOS_ID = 2
EOS_ID = 3

print("Paramètres COFI chargés.")
print("Taille du vocabulaire :", VOCAB_SIZE)
print("Longueur encodeur :", MAX_LEN_ENCODER)
print("Longueur cible complète :", MAX_LEN_TARGET)
print("Longueur décodeur :", MAX_LEN_DECODER)
print("Tokens spéciaux :", PAD_ID, UNK_ID, SOS_ID, EOS_ID)

Paramètres COFI chargés.
Taille du vocabulaire : 38125
Longueur encodeur : 60
Longueur cible complète : 62
Longueur décodeur : 61
Tokens spéciaux : 0 1 2 3


In [21]:
train_charge = np.load(DOSSIER_PROCESSED / "train_prepared.npz")
validation_charge = np.load(DOSSIER_PROCESSED / "validation_prepared.npz")
test_charge = np.load(DOSSIER_PROCESSED / "test_prepared.npz")

X_train = train_charge["encoder"]
decoder_train = train_charge["decoder_input"]
Y_train = train_charge["decoder_target"]

X_val = validation_charge["encoder"]
decoder_val = validation_charge["decoder_input"]
Y_val = validation_charge["decoder_target"]

X_test = test_charge["encoder"]
decoder_test = test_charge["decoder_input"]
Y_test = test_charge["decoder_target"]

print("Données COFI chargées avec succès.")
print("Entraînement :", X_train.shape, decoder_train.shape, Y_train.shape)
print("Validation :", X_val.shape, decoder_val.shape, Y_val.shape)
print("Test :", X_test.shape, decoder_test.shape, Y_test.shape)

Données COFI chargées avec succès.
Entraînement : (53649, 60) (53649, 61) (53649, 61)
Validation : (6707, 60) (6707, 61) (6707, 61)
Test : (6707, 60) (6707, 61) (6707, 61)


# Sprint 3 : Architecture du modèle COFI (Seq2Seq LSTM)

**Objectif** : construire le modèle encodeur-décodeur LSTM, sans attention dans un premier temps, et vérifier qu'il accepte nos séquences et produit une sortie de la bonne forme, même sans entraînement.

Ce notebook est indépendant du sprint 2 : tant que le vocabulaire et la longueur maximale ne sont pas finalisés, on travaille avec des **valeurs provisoires** (section 1) et des **données aléatoires** (section 4). Quand le sprint 2 sera terminé, il suffira de remplacer les valeurs de la section 1.

## 0. Les notions à comprendre

- **RNN** : un réseau qui lit une séquence élément par élément en gardant une mémoire de ce qu'il a lu.
- **LSTM** : un RNN dont la mémoire ne s'efface pas sur les longues séquences, grâce à des portes (oubli, entrée, sortie) et à une mémoire longue `c`. Un RNN simple oublie le début d'une phrase longue.
- **Encodeur** : lit la phrase fautive et la résume dans deux vecteurs d'états (`h` et `c`).
- **Décodeur** : part de ce résumé et écrit la phrase corrigée, un token à la fois.
- **Embedding** : transforme l'identifiant d'un token en un petit vecteur de nombres, appris pendant l'entraînement.
- **Teacher forcing** : pendant l'entraînement, le décodeur reçoit en entrée la vraie phrase corrigée décalée d'un cran, pour apprendre à prédire le token suivant.

## 1. Paramètres (à remplacer par ceux du sprint 2)

In [22]:
from pathlib import Path

# Recherche automatique de la racine du projet COFI
candidats = [Path.cwd(), *Path.cwd().parents]

RACINE_PROJET = next(
    (
        dossier for dossier in candidats
        if (dossier / "data").is_dir()
        and (dossier / "notebooks").is_dir()
    ),
    None
)

if RACINE_PROJET is None:
    raise FileNotFoundError("Racine du projet COFI introuvable.")

# Chemin vers les données préparées
DOSSIER_PROCESSED = RACINE_PROJET / "data" / "processed"

print("Racine du projet :", RACINE_PROJET)
print("Dossier des données :", DOSSIER_PROCESSED)

Racine du projet : /home/nana-marc/Documents/coursBazongo/projetdeep/COFI
Dossier des données : /home/nana-marc/Documents/coursBazongo/projetdeep/COFI/data/processed


In [23]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import json

# ==================================================
# 1. Chargement des paramètres issus du Sprint 2
# ==================================================

with open(DOSSIER_PROCESSED / "parametres_preparation.json",
          "r", encoding="utf-8") as fichier:
    parametres = json.load(fichier)

SRC_VOCAB_SIZE = parametres["taille_vocabulaire"]
TGT_VOCAB_SIZE = parametres["taille_vocabulaire"]

MAX_LEN_SRC = parametres["longueur_max_encodeur"]
MAX_LEN_TGT = parametres["longueur_max_cible_complete"]
MAX_LEN_DECODER = parametres["longueur_decodeur"]

# ==================================================
# 2. Identifiants des tokens spéciaux
# ==================================================

PAD_ID = 0
UNK_ID = 1
SOS_ID = 2
EOS_ID = 3

# ==================================================
# 3. Hyperparamètres de l'architecture
# ==================================================

EMB_DIM = 128
UNITS = 256

# ==================================================
# 4. Reproductibilité
# ==================================================

keras.utils.set_random_seed(42)

# ==================================================
# 5. Vérification
# ==================================================

print("TensorFlow :", tf.__version__)
print("Vocabulaire source :", SRC_VOCAB_SIZE)
print("Vocabulaire cible :", TGT_VOCAB_SIZE)
print("Longueur encodeur :", MAX_LEN_SRC)
print("Longueur cible complète :", MAX_LEN_TGT)
print("Longueur décodeur :", MAX_LEN_DECODER)
print("Tokens spéciaux :", PAD_ID, UNK_ID, SOS_ID, EOS_ID)
print("Embedding :", EMB_DIM)
print("Unités LSTM :", UNITS)

TensorFlow : 2.21.0
Vocabulaire source : 38125
Vocabulaire cible : 38125
Longueur encodeur : 60
Longueur cible complète : 62
Longueur décodeur : 61
Tokens spéciaux : 0 1 2 3
Embedding : 128
Unités LSTM : 256


**Point à confirmer avec le collègue** : la valeur de `PAD_ID` doit être **0**. Le modèle utilise `mask_zero=True`, donc l'identifiant 0 est ignoré (il sert au padding). Si son vocabulaire donne un autre identifiant à `<PAD>`, il faut le lui faire changer.

## 2. Construction du modèle d'entraînement

Le modèle a **deux entrées** et **une sortie** :

| Entrée / sortie | Contenu | Forme |
|---|---|---|
| `encoder_input` | phrase incorrecte (identifiants, avec padding) | (lot, MAX_LEN_SRC) |
| `decoder_input` | phrase correcte décalée : `<SOS> t1 t2 ...` | (lot, MAX_LEN_TGT) |
| sortie | probabilité de chaque token du vocabulaire à chaque position | (lot, MAX_LEN_TGT, TGT_VOCAB_SIZE) |

In [24]:
# ==================================================
# 1. ENCODEUR : lit la phrase incorrecte
# ==================================================

encoder_inputs = keras.Input(
    shape=(MAX_LEN_SRC,),
    dtype="int32",
    name="encoder_input"
)

# Transformation des identifiants en vecteurs
encoder_emb = layers.Embedding(
    input_dim=SRC_VOCAB_SIZE,
    output_dim=EMB_DIM,
    mask_zero=True,
    name="encoder_embedding"
)(encoder_inputs)

# LSTM de l'encodeur
_, state_h, state_c = layers.LSTM(
    UNITS,
    return_state=True,
    name="encoder_lstm"
)(encoder_emb)


# ==================================================
# 2. DÉCODEUR : génère la phrase corrigée
# ==================================================

decoder_inputs = keras.Input(
    shape=(MAX_LEN_DECODER,),
    dtype="int32",
    name="decoder_input"
)

# Embedding du décodeur
decoder_embedding = layers.Embedding(
    input_dim=TGT_VOCAB_SIZE,
    output_dim=EMB_DIM,
    mask_zero=True,
    name="decoder_embedding"
)

decoder_emb = decoder_embedding(decoder_inputs)

# LSTM du décodeur initialisé avec les états de l'encodeur
decoder_lstm = layers.LSTM(
    UNITS,
    return_sequences=True,
    return_state=True,
    name="decoder_lstm"
)

decoder_seq, _, _ = decoder_lstm(
    decoder_emb,
    initial_state=[state_h, state_c]
)


# ==================================================
# 3. COUCHE DE SORTIE : prédit les tokens
# ==================================================

decoder_dense = layers.Dense(
    TGT_VOCAB_SIZE,
    activation="softmax",
    name="output_dense"
)

decoder_outputs = decoder_dense(decoder_seq)


# ==================================================
# 4. CONSTRUCTION DU MODÈLE COMPLET
# ==================================================

model = keras.Model(
    inputs=[encoder_inputs, decoder_inputs],
    outputs=decoder_outputs,
    name="cofi_seq2seq"
)

# Résumé de l'architecture
model.summary()

Model: "cofi_seq2seq"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_input       │ (None, 60)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_input       │ (None, 61)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_embedding   │ (None, 60, 128)   │  4,880,000 │ encoder_input[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_3         │ (None, 60)        │          0 │ encoder_input[0]… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_embedding   │ (None, 61, 128)   │  4,880,000 │ decoder_input[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm (LSTM) │ [(None, 256),     │    394,240 │ encoder_embeddin… │
│                     │ (None, 256),      │            │ not_equal_3[0][0] │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm (LSTM) │ [(None, 61, 256), │    394,240 │ decoder_embeddin… │
│                     │ (None, 256),      │            │ encoder_lstm[0][… │
│                     │ (None, 256)]      │            │ encoder_lstm[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output_dense        │ (None, 61, 38125) │  9,798,125 │ decoder_lstm[0][… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 20,346,605 (77.62 MB)

 Trainable params: 20,346,605 (77.62 MB)

 Non-trainable params: 0 (0.00 B)

### Pourquoi ce nombre de paramètres ?

Chaque couche contribue ainsi (avec `E = EMB_DIM`, `U = UNITS`, `V` = taille du vocabulaire) :

- **Embedding** : `V x E` (un vecteur de taille E par token).
- **LSTM** : `4 x (E + U + 1) x U` (4 portes, chacune avec ses poids d'entrée, de mémoire et son biais).
- **Dense** : `U x V + V` (poids + biais pour chaque token du vocabulaire).

Les couches **Embedding** et **Dense** pèsent le plus quand le vocabulaire est grand : c'est la raison pour laquelle on limite la taille du vocabulaire (par exemple aux 20 000 mots les plus fréquents, le reste devenant `<UNK>`). Le calcul ci-dessous le vérifie.

In [25]:
E, U = EMB_DIM, UNITS
emb_enc  = SRC_VOCAB_SIZE * E
emb_dec  = TGT_VOCAB_SIZE * E
lstm     = 4 * (E + U + 1) * U       # même formule pour l'encodeur et le décodeur
dense    = U * TGT_VOCAB_SIZE + TGT_VOCAB_SIZE
total_calcule = emb_enc + emb_dec + 2 * lstm + dense
print("Calculé à la main :", f"{total_calcule:,}")
print("Donné par Keras   :", f"{model.count_params():,}")
assert total_calcule == model.count_params()

Calculé à la main : 20,346,605
Donné par Keras   : 20,346,605


## 3. Vérification : les poids sont initialisés aléatoirement

Aucun modèle préentraîné n'est chargé : tous les poids sont tirés au hasard à la création. Les poids de l'embedding et du LSTM sont donc des nombres sans signification, de moyenne proche de 0.

In [26]:
for nom in [
    "encoder_embedding",
    "encoder_lstm",
    "decoder_lstm",
    "output_dense"
]:
    w = model.get_layer(nom).get_weights()[0]

    print(
        f"{nom:20s} "
        f"forme {str(w.shape):15s} "
        f"moyenne {w.mean():+.4f} "
        f"écart-type {w.std():.4f}"
    )

encoder_embedding    forme (38125, 128)    moyenne -0.0000 écart-type 0.0289
encoder_lstm         forme (128, 1024)     moyenne -0.0003 écart-type 0.0417
decoder_lstm         forme (128, 1024)     moyenne -0.0000 écart-type 0.0417
output_dense         forme (256, 38125)    moyenne +0.0000 écart-type 0.0072


## 4. Vérification : forme de la sortie, sans entraînement

On fabrique de **fausses données** de la bonne forme (identifiants aléatoires, avec du padding à la fin) et on vérifie ce que le modèle produit.

In [27]:
# ==================================================
# TEST DU MODÈLE AVEC DES DONNÉES SIMULÉES
# ==================================================

BATCH = 4

rng = np.random.default_rng(0)

# --------------------------------------------------
# 1. Entrée de l'encodeur : 4 phrases de 60 tokens
# --------------------------------------------------

x_enc = rng.integers(
    4,
    SRC_VOCAB_SIZE,
    size=(BATCH, MAX_LEN_SRC)
)

# Ajout de padding à la fin
x_enc[:, -6:] = PAD_ID


# --------------------------------------------------
# 2. Entrée du décodeur : 4 séquences de 61 tokens
# --------------------------------------------------

x_dec = rng.integers(
    4,
    TGT_VOCAB_SIZE,
    size=(BATCH, MAX_LEN_DECODER)
)

# Le premier token doit être SOS
x_dec[:, 0] = SOS_ID

# Ajout de padding à la fin
x_dec[:, -4:] = PAD_ID


# --------------------------------------------------
# 3. Prédiction
# --------------------------------------------------

sortie = model.predict(
    [x_enc, x_dec],
    verbose=0
)


# --------------------------------------------------
# 4. Vérifications
# --------------------------------------------------

print("Forme de la sortie :", sortie.shape)

print(
    "Forme attendue     :",
    (BATCH, MAX_LEN_DECODER, TGT_VOCAB_SIZE)
)

print(
    "Somme des probabilités à une position :",
    sortie[0, 0].sum()
)

assert sortie.shape == (
    BATCH,
    MAX_LEN_DECODER,
    TGT_VOCAB_SIZE
)

assert np.allclose(sortie[0, 0].sum(), 1.0, atol=1e-5)

print("\nTest du modèle réussi !")

Forme de la sortie : (4, 61, 38125)
Forme attendue     : (4, 61, 38125)
Somme des probabilités à une position : 0.99999994

Test du modèle réussi !


**Lecture** : à chaque position de la phrase corrigée, le modèle donne une probabilité pour chacun des `TGT_VOCAB_SIZE` tokens. La somme vaut 1 (c'est le rôle du softmax). Le token prédit est celui dont la probabilité est la plus élevée.

## 5. Compilation et perte initiale (contrôle de cohérence)

- **Perte** : `sparse_categorical_crossentropy`. Les cibles sont des identifiants entiers (pas de one-hot), ce qui évite de créer un énorme tableau `(lot, longueur, V)`.
- **Optimiseur** : Adam.

Un modèle aléatoire qui ne sait rien répartit sa probabilité à peu près uniformément sur `V` tokens, donc sa perte initiale doit valoir environ **ln(V)**. Si on trouve une valeur très différente, il y a une erreur dans les données ou dans le modèle.

In [28]:
# ==================================================
# 1. COMPILATION DU MODÈLE
# ==================================================

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy"
)


# ==================================================
# 2. CRÉATION DE CIBLES ARTIFICIELLES DE TEST
# ==================================================

y = rng.integers(
    4,
    TGT_VOCAB_SIZE,
    size=(BATCH, MAX_LEN_DECODER),
    dtype=np.int64
)

# Ajout du token EOS avant les positions PAD
y[:, -5] = EOS_ID

# Ajout de padding à la fin
y[:, -4:] = PAD_ID


# ==================================================
# 3. MASQUE POUR IGNORER LES TOKENS PAD
# ==================================================

poids = (y != PAD_ID).astype(np.float32)


# ==================================================
# 4. ÉVALUATION DU MODÈLE NON ENTRAÎNÉ
# ==================================================

perte = model.evaluate(
    [x_enc, x_dec],
    y,
    sample_weight=poids,
    verbose=0
)


# ==================================================
# 5. AFFICHAGE DES RÉSULTATS
# ==================================================

print("Forme des cibles :", y.shape)
print("Forme attendue :", (BATCH, MAX_LEN_DECODER))

print(f"Perte initiale : {perte:.3f}")
print(f"ln(V) : {np.log(TGT_VOCAB_SIZE):.3f}")

assert y.shape == (BATCH, MAX_LEN_DECODER)

print("\nÉvaluation initiale réussie !")

Forme des cibles : (4, 61)
Forme attendue : (4, 61)
Perte initiale : 10.549
ln(V) : 10.549

Évaluation initiale réussie !


## 6. Modèles d'inférence (pour corriger une vraie phrase, sprint 5)

À l'entraînement, le décodeur reçoit la vraie correction. À l'usage, elle n'existe pas : il faut générer **token par token** :
1. L'encodeur résume la phrase fautive en `(h, c)`.
2. On donne `<SOS>` au décodeur, qui prédit le premier token.
3. On redonne ce token au décodeur avec les états mis à jour, et ainsi de suite jusqu'à `<EOS>` ou la longueur maximale.

On réutilise les **mêmes couches** (donc les mêmes poids) dans deux petits modèles.

In [29]:
# ==================================================
# 1. MODÈLE ENCODEUR POUR L'INFÉRENCE
# ==================================================

encoder_model = keras.Model(
    inputs=encoder_inputs,
    outputs=[state_h, state_c],
    name="cofi_encoder_inference"
)


# ==================================================
# 2. MODÈLE DÉCODEUR POUR L'INFÉRENCE
# ==================================================

dec_state_h = keras.Input(
    shape=(UNITS,),
    name="dec_state_h"
)

dec_state_c = keras.Input(
    shape=(UNITS,),
    name="dec_state_c"
)

dec_token = keras.Input(
    shape=(None,),
    dtype="int32",
    name="dec_token"
)

# Réutilisation de l'embedding et du LSTM déjà construits
d_emb = decoder_embedding(dec_token)

d_seq, new_h, new_c = decoder_lstm(
    d_emb,
    initial_state=[dec_state_h, dec_state_c]
)

d_out = decoder_dense(d_seq)

decoder_model = keras.Model(
    inputs=[dec_token, dec_state_h, dec_state_c],
    outputs=[d_out, new_h, new_c],
    name="cofi_decoder_inference"
)


# ==================================================
# 3. FONCTION DE CORRECTION PAR DÉCODAGE GLOUTON
# ==================================================

def corriger_ids(seq_ids, max_out=MAX_LEN_DECODER):
    """
    Génère une phrase corrigée sous forme d'identifiants.

    Le décodage s'arrête lorsque EOS est généré
    ou lorsque le nombre maximal d'étapes est atteint.
    """

    # Encodage de la phrase incorrecte
    seq_ids = np.asarray(seq_ids, dtype=np.int32)

    h, c = encoder_model.predict(
        seq_ids[None, :],
        verbose=0
    )

    # Le décodeur commence avec SOS
    token = np.array([[SOS_ID]], dtype=np.int32)

    sortie = []

    for _ in range(max_out):

        # Prédiction du prochain token
        probas, h, c = decoder_model.predict(
            [token, h, c],
            verbose=0
        )

        scores = probas[0, -1].copy()

        # Interdire PAD et SOS dans la sortie
        scores[PAD_ID] = -np.inf
        scores[SOS_ID] = -np.inf

        suivant = int(np.argmax(scores))

        # Arrêt si EOS est généré
        if suivant == EOS_ID:
            break

        sortie.append(suivant)

        # Le token prédit devient la prochaine entrée
        token = np.array([[suivant]], dtype=np.int32)

    return sortie


# ==================================================
# 4. TEST INITIAL
# ==================================================

resultat = corriger_ids(x_enc[0])

print("Identifiants générés :", resultat[:10])
print("Nombre de tokens générés :", len(resultat))

Identifiants générés : [9184, 6790, 36017, 6154, 10483, 10483, 19172, 37675, 19172, 16412]
Nombre de tokens générés : 61


## 7. Bilan du sprint 3 (à compléter)

| Rubrique | À compléter |
|---|---|
| Ce que nous avons réalisé | Modèle Seq2Seq LSTM construit, forme de sortie vérifiée, modèles d'inférence prêts |
| Ce que nous avons appris | |
| Ce qui reste à faire | Brancher les vrais vocabulaires et séquences du sprint 2, puis entraîner (sprint 4). L'attention reste optionnelle. |
| Prochaine étape | Sprint 4 : entraînement |


In [30]:
# Vérification de la gestion des poids dans Keras

print("Version de TensorFlow :", tf.__version__)
print("Version de Keras :", keras.__version__)

try:
    perte_masquee = keras.losses.SparseCategoricalCrossentropy(
        reduction="mean_with_sample_weight"
    )

    print("\nRéduction mean_with_sample_weight disponible :", True)
    print("Configuration :", perte_masquee.reduction)

except Exception as erreur:
    print("\nCette méthode n'est pas disponible.")
    print("Erreur :", erreur)

Version de TensorFlow : 2.21.0
Version de Keras : 3.15.1

Réduction mean_with_sample_weight disponible : True
Configuration : mean_with_sample_weight


In [31]:
# Fonction de perte qui tient compte des poids des tokens
perte_masquee = keras.losses.SparseCategoricalCrossentropy(
    reduction="mean_with_sample_weight"
)

# Compilation définitive du modèle
model.compile(
    optimizer=keras.optimizers.Adam(),
    loss=perte_masquee
)

print("Modèle compilé avec succès.")
print("Optimiseur :", model.optimizer.__class__.__name__)
print("Fonction de perte :", model.loss.name)
print("Réduction :", model.loss.reduction)

Modèle compilé avec succès.
Optimiseur : Adam
Fonction de perte : sparse_categorical_crossentropy
Réduction : mean_with_sample_weight


In [32]:
nombre_vrais_tokens = int(tf.reduce_sum(poids_test).numpy())
nombre_total_tokens = int(tf.size(y_test).numpy())
nombre_pad = nombre_total_tokens - nombre_vrais_tokens

print("Nombre de vrais tokens :", nombre_vrais_tokens)
print("Nombre de tokens PAD :", nombre_pad)
print(f"Perte Keras : {float(perte_calculee):.6f}")
print(f"Perte manuelle : {float(perte_manuelle):.6f}")

assert np.allclose(
    float(perte_calculee),
    float(perte_manuelle),
    atol=1e-6
)

print("\nTest du masquage réussi !")

Nombre de vrais tokens : 5
Nombre de tokens PAD : 3
Perte Keras : 2.300905
Perte manuelle : 2.300905

Test du masquage réussi !


In [33]:
# Recharger les données du Sprint 2

fichier_train = np.load(DOSSIER_PROCESSED / "train_prepared.npz")
fichier_val = np.load(DOSSIER_PROCESSED / "validation_prepared.npz")
fichier_test = np.load(DOSSIER_PROCESSED / "test_prepared.npz")

# Entraînement
X_train = fichier_train["encoder"]
decoder_train = fichier_train["decoder_input"]
Y_train = fichier_train["decoder_target"]

# Validation
X_val = fichier_val["encoder"]
decoder_val = fichier_val["decoder_input"]
Y_val = fichier_val["decoder_target"]

# Test
X_test = fichier_test["encoder"]
decoder_test = fichier_test["decoder_input"]
Y_test = fichier_test["decoder_target"]

print("Entraînement :", X_train.shape, decoder_train.shape, Y_train.shape)
print("Validation :", X_val.shape, decoder_val.shape, Y_val.shape)
print("Test :", X_test.shape, decoder_test.shape, Y_test.shape)

Entraînement : (53649, 60) (53649, 61) (53649, 61)
Validation : (6707, 60) (6707, 61) (6707, 61)
Test : (6707, 60) (6707, 61) (6707, 61)


In [35]:
# Fonction de perte avec prise en compte des poids de masquage
perte_masquee = keras.losses.SparseCategoricalCrossentropy(
    reduction="mean_with_sample_weight"
)

# Recompilation du modèle
model.compile(
    optimizer=keras.optimizers.Adam(),
    loss=perte_masquee
)

print("Optimiseur :", model.optimizer.__class__.__name__)
print("Fonction de perte :", model.loss)
print("Réduction :", model.loss.reduction)

Optimiseur : Adam
Fonction de perte : <LossFunctionWrapper(<function sparse_categorical_crossentropy at 0x7cf18a11ac00>, kwargs={'from_logits': False, 'ignore_class': None, 'axis': -1})>
Réduction : mean_with_sample_weight


In [34]:
print("Modèle construit :", "model" in globals())
print("Données train :", "X_train" in globals() and "Y_train" in globals())
print("Données validation :", "X_val" in globals() and "Y_val" in globals())
print("Fonction de perte :", model.loss)

Modèle construit : True
Données train : True
Données validation : True
Fonction de perte : <LossFunctionWrapper(<function sparse_categorical_crossentropy at 0x7cf18a11ac00>, kwargs={'from_logits': False, 'ignore_class': None, 'axis': -1})>


In [36]:
# Création des poids de masquage
poids_train = (Y_train != PAD_ID).astype(np.float32)
poids_val = (Y_val != PAD_ID).astype(np.float32)

# Vérification des dimensions
print("Dimensions poids train :", poids_train.shape)
print("Dimensions poids validation :", poids_val.shape)

# Comptage des vrais tokens et des PAD
print("\nEntraînement")
print("Vrais tokens :", int(poids_train.sum()))
print("Tokens PAD :", int(poids_train.size - poids_train.sum()))

print("\nValidation")
print("Vrais tokens :", int(poids_val.sum()))
print("Tokens PAD :", int(poids_val.size - poids_val.sum()))

Dimensions poids train : (53649, 61)
Dimensions poids validation : (6707, 61)

Entraînement
Vrais tokens : 934880
Tokens PAD : 2337709

Validation
Vrais tokens : 117244
Tokens PAD : 291883


In [37]:
# Vérification des types de données
print("Types des tableaux :")
print("X_train :", X_train.dtype)
print("decoder_train :", decoder_train.dtype)
print("Y_train :", Y_train.dtype)
print("poids_train :", poids_train.dtype)

# Vérification des dimensions
assert X_train.shape[1] == MAX_LEN_SRC
assert decoder_train.shape[1] == MAX_LEN_DECODER
assert Y_train.shape == decoder_train.shape
assert poids_train.shape == Y_train.shape

# Vérification des identifiants
assert X_train.min() >= 0
assert X_train.max() < SRC_VOCAB_SIZE
assert decoder_train.min() >= 0
assert decoder_train.max() < TGT_VOCAB_SIZE
assert Y_train.min() >= 0
assert Y_train.max() < TGT_VOCAB_SIZE

# Vérification des tokens spéciaux
assert np.all(Y_train[:, 0] == EOS_ID) == False
assert np.all(poids_train[Y_train == PAD_ID] == 0)
assert np.all(poids_train[Y_train != PAD_ID] == 1)

print("\nToutes les vérifications sont réussies.")

Types des tableaux :
X_train : int32
decoder_train : int32
Y_train : int32
poids_train : float32

Toutes les vérifications sont réussies.


In [38]:
# Hyperparamètres d'entraînement

BATCH_SIZE = 8
EPOCHS = 20
LEARNING_RATE = 0.001

print("Taille du batch :", BATCH_SIZE)
print("Nombre maximal d'époques :", EPOCHS)
print("Taux d'apprentissage :", LEARNING_RATE)

Taille du batch : 8
Nombre maximal d'époques : 20
Taux d'apprentissage : 0.001


In [39]:
from tensorflow.keras.callbacks import (
    ModelCheckpoint,
    EarlyStopping,
    ReduceLROnPlateau
)

# Dossier de sauvegarde
DOSSIER_MODELS = RACINE_PROJET / "models"
DOSSIER_MODELS.mkdir(parents=True, exist_ok=True)

# Sauvegarde du meilleur modèle
checkpoint = ModelCheckpoint(
    filepath=DOSSIER_MODELS / "cofi_meilleur.weights.h5",
    monitor="val_loss",
    mode="min",
    save_best_only=True,
    save_weights_only=True,
    verbose=1
)

# Arrêt anticipé
early_stopping = EarlyStopping(
    monitor="val_loss",
    mode="min",
    patience=4,
    restore_best_weights=True,
    verbose=1
)

# Réduction du taux d'apprentissage
reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    mode="min",
    factor=0.5,
    patience=2,
    min_lr=0.00001,
    verbose=1
)

callbacks = [
    checkpoint,
    early_stopping,
    reduce_lr
]

print("Callbacks configurés :", len(callbacks))

Callbacks configurés : 3


In [40]:
# Configuration définitive de l'optimiseur
optimiseur = keras.optimizers.Adam(
    learning_rate=LEARNING_RATE
)

model.compile(
    optimizer=optimiseur,
    loss=perte_masquee
)

print("Optimiseur :", model.optimizer.__class__.__name__)
print("Taux d'apprentissage :", float(model.optimizer.learning_rate.numpy()))
print("Fonction de perte :", model.loss.reduction)
print("Nombre de paramètres :", f"{model.count_params():,}")

Optimiseur : Adam
Taux d'apprentissage : 0.0010000000474974513
Fonction de perte : mean_with_sample_weight
Nombre de paramètres : 20,346,605


In [41]:
import psutil

memoire = psutil.virtual_memory()

print("Mémoire RAM totale :", round(memoire.total / (1024**3), 2), "Go")
print("Mémoire RAM disponible :", round(memoire.available / (1024**3), 2), "Go")
print("Mémoire RAM utilisée :", round(memoire.percent, 1), "%")

Mémoire RAM totale : 7.63 Go
Mémoire RAM disponible : 0.95 Go
Mémoire RAM utilisée : 87.5 %
